# M512 Hotel Booking Demand — Forensic EDA v3 Deep Dive

This pass attacks the candidate story before final freeze. It adds duplicate sensitivity, missingness signals, partial-year controls, effect sizes, subgroup stability, categorical redundancy, cancellation timing, no-shows, model interactions and a final sensitivity matrix.

**Conservative rule:** a striking pattern is not promoted unless it survives denominator, timing, subgroup and data-quality checks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
from pathlib import Path

CSV=Path("hotel_bookings.csv")
if not CSV.exists():
    import kagglehub
    folder=Path(kagglehub.dataset_download("jessemostipak/hotel-booking-demand"))
    CSV=next(iter(folder.rglob("hotel_bookings.csv")))
df=pd.read_csv(CSV)

month_num={"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,"July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df["arrival_month_num"]=df.arrival_date_month.map(month_num)
df["arrival_date"]=pd.to_datetime(dict(year=df.arrival_date_year,month=df.arrival_month_num,day=df.arrival_date_day_of_month))
df["reservation_status_date"]=pd.to_datetime(df.reservation_status_date)
df["stay_nights"]=df.stays_in_weekend_nights+df.stays_in_week_nights
df["guests"]=df.adults+df.children.fillna(0)+df.babies
df["has_prior_success"]=df.previous_bookings_not_canceled.gt(0)
df["has_special_request"]=df.total_of_special_requests.gt(0)
bins=[-1,7,30,90,180,365,np.inf]; labels=["0–7","8–30","31–90","91–180","181–365","366+"]
df["lead_band"]=pd.cut(df.lead_time,bins,labels=labels,ordered=True)
df["lead_band_str"]=df.lead_band.astype(str)
print(df.shape,df.arrival_date.min(),df.arrival_date.max())

## 1. Duplicate ambiguity — material sensitivity

In [ ]:
dedup=df.drop_duplicates().copy()

def key_summary(data):
    lead=data.groupby("lead_band",observed=True).is_canceled.mean()*100
    seg=data.groupby("market_segment").is_canceled.agg(["size","sum","mean"])
    return pd.Series({
        "n":len(data),
        "cancel_rate_pct":data.is_canceled.mean()*100,
        "lead_0_7_pct":lead.loc["0–7"],
        "lead_366plus_pct":lead.loc["366+"],
        "groups_pct":seg.loc["Groups","mean"]*100,
        "top3_cancel_share":seg.sort_values("sum",ascending=False).head(3)["sum"].sum()/data.is_canceled.sum()*100
    })
display(pd.DataFrame({"Full":key_summary(df),"drop_duplicates sensitivity":key_summary(dedup)}).T.round(2))

dup_group=df.duplicated(keep=False)
display((pd.crosstab(df.market_segment,dup_group,normalize="index")*100).round(1))
display((pd.crosstab(df.deposit_type,dup_group,normalize="index")*100).round(1))
display((pd.crosstab(df.lead_band,dup_group,normalize="index")*100).round(1))

The source has no unique booking identifier. Therefore exact repeated rows cannot safely be declared errors, but duplicate handling materially changes some magnitudes. Source rows remain primary; deduplication is a sensitivity test.

## 2. Missingness and suspicious combinations

In [ ]:
for col in ["children","country","agent","company"]:
    miss=df[col].isna()
    print("\n",col)
    display(pd.DataFrame({
        "n":[(~miss).sum(),miss.sum()],
        "cancel_rate_pct":[df.loc[~miss,"is_canceled"].mean()*100,df.loc[miss,"is_canceled"].mean()*100]
    },index=["Observed","Missing"]).round(2))

flags=pd.Series({
    "zero_guests":(df.guests==0).sum(),
    "zero_stay_nights":(df.stay_nights==0).sum(),
    "negative_ADR":(df.adr<0).sum(),
    "ADR_zero":(df.adr==0).sum(),
    "ADR_gt_1000":(df.adr>1000).sum(),
    "reserved_assigned_room_mismatch":(df.reserved_room_type!=df.assigned_room_type).sum()
},name="rows")
display(flags.to_frame())

## 3. Partial-year bias — matched July/August comparison

In [ ]:
matched=df[df.arrival_month_num.isin([7,8])]
all_year=df.groupby("arrival_date_year").is_canceled.agg(["size","sum","mean"]); all_year["mean"]*=100
matched_year=matched.groupby("arrival_date_year").is_canceled.agg(["size","sum","mean"]); matched_year["mean"]*=100
print("All available months"); display(all_year.round(1))
print("Matched July-August"); display(matched_year.round(1))

2015 begins in July and 2017 ends in August. Matching July-August prevents raw partial-year composition from being mistaken for a clean annual effect.

## 4. Wilson intervals and effect sizes

In [ ]:
def wilson(successes,n,z=1.96):
    p=successes/n
    denom=1+z*z/n
    center=(p+z*z/(2*n))/denom
    half=z*np.sqrt((p*(1-p)/n)+z*z/(4*n*n))/denom
    return center-half,center+half

def effect(mask_a,mask_b,label_a,label_b):
    A=df[mask_a]; B=df[mask_b]
    pa=A.is_canceled.mean(); pb=B.is_canceled.mean()
    cia=wilson(int(A.is_canceled.sum()),len(A)); cib=wilson(int(B.is_canceled.sum()),len(B))
    return pd.Series({"A":label_a,"n_A":len(A),"rate_A_pct":pa*100,"CI_A_low":cia[0]*100,"CI_A_high":cia[1]*100,
                      "B":label_b,"n_B":len(B),"rate_B_pct":pb*100,"CI_B_low":cib[0]*100,"CI_B_high":cib[1]*100,
                      "risk_difference_pp":(pa-pb)*100,"risk_ratio":pa/pb if pb else np.nan})
display(pd.DataFrame([
    effect(df.hotel.eq("City Hotel"),df.hotel.eq("Resort Hotel"),"City","Resort"),
    effect(df.lead_band.eq("366+"),df.lead_band.eq("0–7"),"366+","0–7"),
    effect(df.market_segment.eq("Groups"),df.market_segment.eq("Direct"),"Groups","Direct"),
    effect(df.has_prior_success,~df.has_prior_success,"Prior success recorded","None recorded"),
    effect(df.has_special_request,~df.has_special_request,"≥1 request","No request")
]).round(2))

## 5. Simpson's-paradox / subgroup stability check

In [ ]:
def lead_profile(data):
    x=data.groupby("lead_band",observed=True).is_canceled.agg(["size","mean"])
    x["rate_pct"]=x["mean"]*100
    return x[["size","rate_pct"]]

for h in df.hotel.unique():
    print("\nHOTEL:",h); display(lead_profile(df[df.hotel.eq(h)]).round(1))
for seg in ["Online TA","Groups","Offline TA/TO","Direct","Corporate"]:
    print("\nSEGMENT:",seg); display(lead_profile(df[df.market_segment.eq(seg)]).round(1))

## 6. Predictor redundancy — Cramér's V

In [ ]:
def cramers_v(a,b):
    table=pd.crosstab(a,b)
    chi2=chi2_contingency(table,correction=False)[0]
    n=table.values.sum(); phi2=chi2/n; r,k=table.shape
    phi2corr=max(0,phi2-((k-1)*(r-1))/(n-1))
    rcorr=r-((r-1)**2)/(n-1); kcorr=k-((k-1)**2)/(n-1)
    den=min(kcorr-1,rcorr-1)
    return np.sqrt(phi2corr/den) if den>0 else np.nan

pairs=[("market_segment","distribution_channel"),("market_segment","customer_type"),("market_segment","deposit_type"),
       ("distribution_channel","deposit_type"),("hotel","market_segment"),("hotel","distribution_channel")]
display(pd.DataFrame([(a,b,cramers_v(df[a].astype(str),df[b].astype(str))) for a,b in pairs],
                     columns=["var1","var2","cramers_v"]).sort_values("cramers_v",ascending=False).round(3))

Market segment and distribution channel overlap heavily. They should not be presented as two independent headline discoveries unless they support different decisions.

## 7. Concentration across segment × lead-time cells

In [ ]:
cells2=(df.groupby(["market_segment","lead_band"],observed=True).is_canceled
          .agg(bookings="size",cancellations="sum",cancel_rate="mean").reset_index())
cells2["cancel_rate"]*=100
cells2=cells2[cells2.bookings>=100].sort_values("cancellations",ascending=False)
cells2["cum_cancel_share_pct"]=cells2.cancellations.cumsum()/df.is_canceled.sum()*100
display(cells2.head(25).round(1))
for k in [3,5,6,10]:
    print(k,round(cells2.head(k).cancellations.sum()/df.is_canceled.sum()*100,1))

## 8. Cancellation timing — probability is not operational harm

In [ ]:
cancel=df[df.reservation_status.eq("Canceled")].copy()
cancel["notice_days"]=(cancel.arrival_date-cancel.reservation_status_date).dt.days
display(cancel.notice_days.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]).to_frame().T)
cancel["late7"]=cancel.notice_days.between(0,7,inclusive="both")
print("Canceled within 0–7 days:",int(cancel.late7.sum()),f"({cancel.late7.mean()*100:.1f}%)")
display(cancel.groupby("lead_band",observed=True).notice_days.agg(n="size",median="median",mean="mean").round(1))

A high cancellation probability is not identical to high operational damage. Earlier cancellation may leave more time to resell inventory, although resale is not observed in this dataset.

## 9. Separate no-shows from advance cancellations

In [ ]:
status=df.reservation_status.value_counts().to_frame("n"); status["share_pct"]=status.n/len(df)*100
display(status.round(2))
display((pd.crosstab(df.market_segment,df.reservation_status,normalize="index")*100).round(1))

late_by_seg=cancel.groupby("market_segment").late7.sum()
noshow=df[df.reservation_status.eq("No-Show")].market_segment.value_counts()
prox=pd.DataFrame({"late_cancel_0_7":late_by_seg,"no_show":noshow}).fillna(0)
prox["arrival_prox_fail"]=prox.sum(axis=1)
prox["share_pct"]=prox.arrival_prox_fail/prox.arrival_prox_fail.sum()*100
display(prox.sort_values("arrival_prox_fail",ascending=False).round(1))

## 10. Adjusted model with hotel × segment interaction

In [ ]:
model_df=df[df.market_segment.ne("Undefined")].copy()
base_formula=("is_canceled ~ C(hotel) + C(lead_band_str) + C(market_segment) + has_prior_success + has_special_request + C(customer_type) + C(arrival_month_num) + C(arrival_date_year)")
int_formula=("is_canceled ~ C(hotel) * C(market_segment) + C(lead_band_str) + has_prior_success + has_special_request + C(customer_type) + C(arrival_month_num) + C(arrival_date_year)")
base=smf.glm(base_formula,data=model_df,family=sm.families.Binomial()).fit()
inter=smf.glm(int_formula,data=model_df,family=sm.families.Binomial()).fit()
print("Base AIC",round(base.aic,1),"Interaction AIC",round(inter.aic,1),"Improvement",round(base.aic-inter.aic,1))
ci=inter.conf_int()
odds=pd.DataFrame({"odds_ratio":np.exp(inter.params),"ci_low":np.exp(ci[0]),"ci_high":np.exp(ci[1]),"p_value":inter.pvalues})
focus=[i for i in odds.index if any(k in i for k in ["lead_band_str","market_segment","has_prior_success","has_special_request","hotel"])]
display(odds.loc[focus].round(3))

The interaction model remains associational. It tests robustness and heterogeneity; it does not establish policy effects.

## 11. Final sensitivity matrix

In [ ]:
quality=df[(df.guests>0)&(df.stay_nights>0)&(df.adr>=0)&(df.adr<=df.adr.quantile(.995))].copy()
pops={"Full":df,"Drop exact duplicate-looking rows":dedup,"Quality-filtered":quality}
rows=[]
for name,d in pops.items():
    lead=d.groupby("lead_band",observed=True).is_canceled.mean()*100
    seg=d.groupby("market_segment").is_canceled.agg(["sum","mean"])
    rows.append({
        "population":name,"n":len(d),"overall_pct":d.is_canceled.mean()*100,
        "lead_0_7_pct":lead.loc["0–7"],"lead_366plus_pct":lead.loc["366+"],
        "top3_cancel_share_pct":seg.sort_values("sum",ascending=False).head(3)["sum"].sum()/d.is_canceled.sum()*100,
        "groups_city_pct":d[(d.market_segment.eq("Groups"))&(d.hotel.eq("City Hotel"))].is_canceled.mean()*100,
        "groups_resort_pct":d[(d.market_segment.eq("Groups"))&(d.hotel.eq("Resort Hotel"))].is_canceled.mean()*100
    })
display(pd.DataFrame(rows).round(2))

## 12. v3 promotion gate

**KEEP:** cancellation burden; lead-time gradient; top-three segment concentration; hotel × segment heterogeneity.

**KEEP WITH SOURCE CAVEAT:** recorded prior successful-booking history.

**ELEVATE:** cancellation timing / arrival-proximate failure because it separates probability from operational timing risk.

**RESERVE:** special requests, distribution channel, booked-value proxies.

**QUALITY ONLY:** deposit type and parking.

**REJECT AS HEADLINE:** previous cancellations, booking changes and ADR bands.

### Deep-dive conclusion

The strongest story is not simply “who cancels most?” Booking failure has two dimensions: **likelihood and timing**. Long-lead and some segment profiles are less reliable, while late cancellations and no-shows create a distinct arrival-proximate operational risk. Management should test targeted controls using both dimensions rather than relying on cancellation rate alone.